### U24AI070
### Dev Sangam Kumar

In [1]:
import os

os.environ['KAGGLE_USERNAME'] = "devsvnit"
os.environ['KAGGLE_KEY'] = "KGAT_ead91e9f0b9c365710cbd699a8f591f3"
os.environ["KAGGLE_API_TOKEN"] = "KGAT_ead91e9f0b9c365710cbd699a8f591f3"
os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
with open(os.path.expanduser("~/.kaggle/access_token"), "w") as f:
  f.write("KGAT_ead91e9f0b9c365710cbd699a8f591f3")

!pip install -q kaggle

In [2]:
import kagglehub

path = kagglehub.dataset_download("devsvnit/hindi-indiccorp-tokenized")

print("Path to dataset files:", path)

100%|██████████| 17.6G/17.6G [13:57<00:00, 22.6MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/devsvnit/hindi-indiccorp-tokenized/versions/1


In [3]:
import pandas as pd
import os
import pyarrow.parquet as pq
import numpy as np

try:
    path
except NameError:
    import kagglehub
    path = kagglehub.dataset_download("devsvnit/hindi-indiccorp-tokenized")

parquet_path = os.path.join(path, 'words.parquet')

total_rows_needed = 1002000
pf = pq.ParquetFile(parquet_path)

rows = []
for batch in pf.iter_batches(batch_size=100000):
    rows.append(batch.to_pandas())
    current_count = sum(len(b) for b in rows)
    if current_count >= total_rows_needed:
        break

df = pd.concat(rows).iloc[:total_rows_needed]

test_set = df.iloc[:1000]["tokens"].to_numpy()
dev_set = df.iloc[1000:2000]["tokens"].to_numpy()
train_set = df.iloc[2000:]["tokens"]

print(f"Train set size: {len(train_set)}")
print(f"Test set size: {len(test_set)}")
print(f"Dev set size: {len(dev_set)}")


Train set size: 1000000
Test set size: 1000
Dev set size: 1000


In [4]:
display(train_set)
display(train_set[0])

,tokens
2000,"[उन्होने, कहा, कि, मुख्यमंत्री, कैप्टन, अमरिंद..."
2001,"[‘, अगर, और, कुछ, नही, कांग्रेस, सरकार, को, सं..."
2002,"[कांग्रेस, सरकार, द्वारा, निजी, अस्पतालों, में..."
2003,"[सरकार, ब्लाॅक, स्तर, पर, कोविड, केंद्र, खोलने..."
2004,"[सरदार, बादल, ने, शिरोमणी, अकाली, दल, के, कार्..."
...,...
1995,"[भारतीय, टीम, 1936, के, ओलंपिक, खेलों, से, पहल..."
1996,"[2019, की, जनवरी, तक, नरेश, ने, अपना, पद, संभा..."
1997,"[लेकिन, जेट, एयरवेज, को, हाथ, से, छोड़, चुके, ..."
1998,"[कंपनी, धीरे, -, धीरे, खत्म, हो, रही, थी।]"


,tokens
0,"[कलक्ट्रेट, परिसर, में, ही, नया, भवन, बनाने, क..."
0,"[मियावाकी, पद्धति, से, सितंबर, 2020, में, तत्क..."
0,"[टीचर, की, बेटी, का, चेस, में, कमाल]"
0,"[गौरतलब, है, कि, पाकिस्तान, ने, अपने, नीकिया, ..."
0,"[300, सीटों, के, लिए, परीक्षाः]"
0,"[संभल, के, हॉट, स्पॉट, सरायतरीन, और, दीपा, सरा..."
0,"[तेजस्वी, यादव, ने, कहा, था, ,, आरजेडी, के, को..."
0,"[मुरैना, के, जौरा, से, बीजेपी, विधायक, सत्यपाल..."
0,"[नई, दिल्ली, .]"
0,"[ऐसे, में, जाहिर, सी, बात, है, कि, वह, पढ़ाई, ..."


In [5]:

train_list = train_set.tolist()

for i in range(len(train_list)):
    train_list[i] = ["<s>", "<s>", "<s>"] + list(train_list[i]) + ["</s>", "</s>", "</s>"]

train_set = np.array(train_list, dtype=object)

print(train_set[0])

['<s>', '<s>', '<s>', 'उन्होने', 'कहा', 'कि', 'मुख्यमंत्री', 'कैप्टन', 'अमरिंदर', 'सिंह', 'के', 'साथ', 'साथ', 'उनके', 'मंत्रिमंडल', 'के', 'सहयोगी', 'लापता', 'हो', 'गए', 'हैं', 'यां', 'दिल्ली', 'की', 'ओर', 'रवाना', 'हो', 'गए', 'हैं।', '</s>', '</s>', '</s>']


In [6]:
uni_grams= {}
bi_grams= {}
tri_grams= {}
quad_grams= {}

for i in range(len(train_set)):
  for j in range(len(train_set[i])):
    if train_set[i][j] not in uni_grams:
      uni_grams[train_set[i][j]] = 1
    else:
      uni_grams[train_set[i][j]] += 1
    if j > 1:
      if (train_set[i][j-2], train_set[i][j-1]) not in bi_grams:
        bi_grams[(train_set[i][j-2], train_set[i][j-1])] = 1
      else:
        bi_grams[(train_set[i][j-2], train_set[i][j-1])] += 1
    if j > 2:
      if (train_set[i][j-2], train_set[i][j-1], train_set[i][j-2]) not in tri_grams:
        tri_grams[(train_set[i][j-2], train_set[i][j-1], train_set[i][j-2])] = 1
      else:
        tri_grams[(train_set[i][j-2], train_set[i][j-1], train_set[i][j-2])] += 1
    if j > 3:
      if (train_set[i][j-2], train_set[i][j-1], train_set[i][j-2] ,train_set[i][j-3]) not in quad_grams:
        quad_grams[(train_set[i][j-2], train_set[i][j-1], train_set[i][j-2] ,train_set[i][j-3])] = 1
      else:
        quad_grams[(train_set[i][j-2], train_set[i][j-1], train_set[i][j-2] ,train_set[i][j-3])] += 1

del uni_grams["<s>"]
del uni_grams["</s>"]
del bi_grams[("<s>", "<s>")]
del bi_grams[("</s>", "</s>")]
del tri_grams[("<s>", "<s>", "<s>")]
del tri_grams[("</s>", "</s>", "</s>")]

print(f"Unique uni-grams: {len(uni_grams)}")
print(f"Unique bi-grams: {len(bi_grams)}")
print(f"Unique tri-grams: {len(tri_grams)}")
print(f"Unique quad-grams: {len(quad_grams)}")


Unique uni-grams: 312111
Unique bi-grams: 3486305
Unique tri-grams: 3486305
Unique quad-grams: 9703119


In [7]:
total_uni_grams= sum(uni_grams.values())
def prob_next_token(history= None, next_token= None):
  if history == None:
    return uni_grams[next_token] / sum(uni_grams.values())
  if len(history) == 1:
    return bi_grams[(history[0], next_token)] / uni_grams[history[0]]
  if len(history) == 2:
    return tri_grams[(history[0], history[1], next_token)] / bi_grams[(history[0], history[1])]
  if len(history) == 3:
    return quad_grams[(history[0], history[1], history[2], next_token)] / tri_grams[(history[0], history[1], history[2])]

In [8]:
import math

def evaluate_all_models(data, uni_counts, bi_counts, tri_counts, quad_counts):
    vocab_size = len(uni_counts)
    models = {1: {'log_prob': 0, 'words': 0},
              2: {'log_prob': 0, 'words': 0},
              3: {'log_prob': 0, 'words': 0},
              4: {'log_prob': 0, 'words': 0}}

    total_uni_sum = sum(uni_counts.values())

    for sentence in data:
        padded = ["<s>"] * 3 + list(sentence) + ["</s>"] * 3

        for i in range(3, len(padded)):
            target = padded[i]

            #unigram
            p1 = (uni_counts.get(target, 0) + 1) / (total_uni_sum + vocab_size)
            models[1]['log_prob'] += math.log2(p1)
            models[1]['words'] += 1

            #bigram
            ctx2 = padded[i-1]
            p2 = (bi_counts.get((ctx2, target), 0) + 1) / (uni_counts.get(ctx2, 0) + vocab_size)
            models[2]['log_prob'] += math.log2(p2)
            models[2]['words'] += 1

            #trigram
            ctx3 = (padded[i-2], padded[i-1])
            p3 = (tri_counts.get((ctx3[0], ctx3[1], target), 0) + 1) / (bi_counts.get(ctx3, 0) + vocab_size)
            models[3]['log_prob'] += math.log2(p3)
            models[3]['words'] += 1

            #quadgram
            ctx4 = (padded[i-3], padded[i-2], padded[i-1])
            p4 = (quad_counts.get((ctx4[0], ctx4[1], ctx4[2], target), 0) + 1) / (tri_counts.get(ctx4, 0) + vocab_size)
            models[4]['log_prob'] += math.log2(p4)
            models[4]['words'] += 1

    perplexities = {}
    for n, stats in models.items():
        avg_lp = stats['log_prob'] / stats['words']
        perplexities[n] = math.pow(2, -avg_lp)

    return perplexities

results = evaluate_all_models(test_set, uni_grams, bi_grams, tri_grams, quad_grams)
for n, perp in results.items():
    print(f"{n}-gram Perplexity: {perp:.2f}")

1-gram Perplexity: 7193.38
2-gram Perplexity: 5734.37
3-gram Perplexity: 310634.06
4-gram Perplexity: 311710.83
